In [36]:
import pandas as pd
import json
import glob

In [37]:
results_dir = '../results'

In [89]:
all_results = glob.glob('../results/*.json')

table_data = []
for result in all_results:
    with open(result, 'r') as f:
        data = json.load(f)
        for k, v in data.items():
            if k in ['Accuracy', 'IoU', 'AP']:
                data[k] = v['mean']        
            if k == 'train_config':
                v = str(v)
                if '[' in v:
                    data[k] = 15
                elif '5' in v:
                    data[k] = 5
                else:
                    data[k] = 2
            if k == 'window_len':
                data['window_len'] = int(v)
        table_data.append(data)

In [90]:
df = pd.DataFrame(table_data)
df.head()

,AP,IoU,Accuracy,number_of_samples,depth_channel_representation_mode,window_len,train_config,multiple_digits,model_path
0,0.206345,0.221674,0.919198,1907,original,20,2,False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...
1,0.359727,0.455038,0.953592,1907,original,20,5,False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...
2,0.423964,0.570064,0.962117,1907,original,20,15,False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...
3,0.173465,0.164156,0.922380,2525,original,10,2,False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...
4,0.190495,0.204647,0.929361,2525,original,10,2,False,C:\Users\Emiel\Documents\TU\dsait4205-ev-mask-...


In [91]:
metrics = ['Accuracy', 'IoU', 'AP']
grouped = df.groupby(['window_len','train_config','multiple_digits','depth_channel_representation_mode'])
means = grouped[metrics].mean(numeric_only=True)
stds = grouped[metrics].std(numeric_only=True)
stds.rename(columns=lambda x: x + '_std', inplace=True)
counts = pd.DataFrame(grouped.size(), columns=['n'])

combined = pd.concat([means, stds, counts], axis=1)
for metric in metrics:
    combined[metric] = combined.apply(lambda x: f"{x[metric]*100:.2f} ± {x[f'{metric}_std']*100:.2f}", axis=1)
    combined.drop([f'{metric}_std'], axis=1, inplace=True)
combined.to_csv('results.csv', index=True)
combined

Accuracy  \
window_len train_config multiple_digits depth_channel_representation_mode                 
10         2            False           original                           93.37 ± 0.71   
           5            False           original                           94.93 ± 0.29   
           15           False           original                           96.06 ± 0.11   
20         2            False           original                           93.16 ± 1.18   
           5            False           original                           95.25 ± 0.18   
           15           False           original                           96.18 ± 0.04   
50         2            False           original                           90.39 ± 3.63   
           5            False           original                           93.10 ± 3.97   
           15           False           original                           96.14 ± 0.05   

                                                                                     IoU  \
window_len train_config multiple_digits depth_channel_representation_mode                  
10         2            False           original                            21.08 ± 3.31   
           5            False           original                            38.55 ± 3.30   
           15           False           original                            55.21 ± 2.05   
20         2            False           original                            19.18 ± 4.86   
           5            False           original                            42.46 ± 2.27   
           15           False           original                            55.93 ± 1.22   
50         2            False           original                            17.01 ± 7.61   
           5            False           original                           35.63 ± 14.81   
           15           False           original                            55.23 ± 1.05   

                                                                                      AP  \
window_len train_config multiple_digits depth_channel_representation_mode                  
10         2            False           original                            19.92 ± 2.34   
           5            False           original                            31.79 ± 2.09   
           15           False           original                            40.23 ± 1.05   
20         2            False           original                            18.65 ± 3.26   
           5            False           original                            33.99 ± 1.40   
           15           False           original                            41.53 ± 0.76   
50         2            False           original                            18.12 ± 6.83   
           5            False           original                           29.22 ± 12.22   
           15           False           original                            42.25 ± 0.76   

                                                                           n  
window_len train_config multiple_digits depth_channel_representation_mode     
10         2            False           original                           6  
           5            False           original                           6  
           15           False           original                           6  
20         2            False           original                           6  
           5            False           original                           6  
           15           False           original                           6  
50         2            False           original                           6  
           5            False           original                           6  
           15           False           original                           5

# Reference results

In [92]:
paper_results = """
window_len,train_config,multiple_digits,depth_channel_representation_mode,Accuracy,IoU,AP
10,2,False,original,93.33,14.19,13.4
10,5,False,original,95.04,41.05,32.8
10,15,False,original,95.64,55.47,42.3
20,2,False,original,94.23,20.48,18.7
20,5,False,original,95.63,47.24,37.1
20,15,False,original,96.29,58.01,43.2
50,2,False,original,94.76,27.82,23.7
50,5,False,original,95.27,41.73,35.2
50,15,False,original,96.51,60.29,44.6
"""
import io
paper_df = pd.read_csv(io.StringIO(paper_results))
paper_df = paper_df.groupby(['window_len','train_config','multiple_digits','depth_channel_representation_mode']).mean() / 100
paper_df.head()

Accuracy  \
window_len train_config multiple_digits depth_channel_representation_mode             
10         2            False           original                             0.9333   
           5            False           original                             0.9504   
           15           False           original                             0.9564   
20         2            False           original                             0.9423   
           5            False           original                             0.9563   

                                                                              IoU  \
window_len train_config multiple_digits depth_channel_representation_mode           
10         2            False           original                           0.1419   
           5            False           original                           0.4105   
           15           False           original                           0.5547   
20         2            False           original                           0.2048   
           5            False           original                           0.4724   

                                                                              AP  
window_len train_config multiple_digits depth_channel_representation_mode         
10         2            False           original                           0.134  
           5            False           original                           0.328  
           15           False           original                           0.423  
20         2            False           original                           0.187  
           5            False           original                           0.371

# T-Test

In [ ]:
from scipy import stats
import numpy as np

# Ensure all DataFrames have the same index
common_index = paper_df.index.intersection(means.index)

# Perform 1-sample t-test for each metric
metrics = ['Accuracy', 'IoU', 'AP']
t_test_results = {}

print("T-test Results:")
print("H0: Our results = Paper results")
print("H1: Our results ≠ Paper results")
print("\nSignificance levels: * p<0.05, ** p<0.01, *** p<0.001")
print("\nResults (t-statistic, p-value):")

for metric in metrics:
    print(f"\n--- {metric} ---")
    t_test_results[metric] = {}
    
    for idx in common_index:
        # Get our sample data (mean and std)
        our_mean = means.loc[idx, metric]
        our_std = stds.loc[idx, f'{metric}_std']
        
        # Get paper (expected) mean
        paper_mean = paper_df.loc[idx, metric]
        
        n = 6  # You can adjust this if you have the actual sample sizes
        
        # Calculate t-statistic for 1-sample t-test
        # t = (sample_mean - population_mean) / (sample_std / sqrt(n))
        t_stat = (our_mean - paper_mean) / (our_std / np.sqrt(n))
        
        # Calculate p-value (two-tailed test)
        p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n-1))
        
        # Store results
        t_test_results[metric][idx] = {
            't_statistic': t_stat,
            'p_value': p_value,
            'our_mean': our_mean,
            'paper_mean': paper_mean,
            'our_std': our_std
        }
        
        # Determine significance level
        if p_value < 0.001:
            sig = "***"
        elif p_value < 0.01:
            sig = "**"
        elif p_value < 0.05:
            sig = "*"
        else:
            sig = ""
        
        print(f"{idx}: t={t_stat:.3f}, p={p_value:.3f}{sig}")
        print(f"  Our: {our_mean:.2f}±{our_std:.2f}% vs Paper: {paper_mean:.2f}%")

# Summary statistics
print("\n" + "="*50)
print("SUMMARY")
print("="*50)

for metric in metrics:
    p_values = [result['p_value'] for result in t_test_results[metric].values()]
    significant_count = sum(1 for p in p_values if p < 0.05)
    
    print(f"\n{metric}:")
    print(f"  Total comparisons: {len(p_values)}")
    print(f"  Significant differences (p<0.05): {significant_count}")
    print(f"  Non-significant: {len(p_values) - significant_count}")

T-test Results:
H0: Our results = Paper results
H1: Our results ≠ Paper results

Significance levels: * p<0.05, ** p<0.01, *** p<0.001

Results (t-statistic, p-value):

--- Accuracy ---
(10, 2, False, 'original'): t=0.146, p=0.889
  Our: 0.93±0.01% vs Paper: 0.93%
(10, 5, False, 'original'): t=-0.931, p=0.394
  Our: 0.95±0.00% vs Paper: 0.95%
(10, 15, False, 'original'): t=9.427, p=0.000***
  Our: 0.96±0.00% vs Paper: 0.96%
(20, 2, False, 'original'): t=-2.235, p=0.076
  Our: 0.93±0.01% vs Paper: 0.94%
(20, 5, False, 'original'): t=-5.168, p=0.004**
  Our: 0.95±0.00% vs Paper: 0.96%
(20, 15, False, 'original'): t=-6.431, p=0.001**
  Our: 0.96±0.00% vs Paper: 0.96%
(50, 2, False, 'original'): t=-2.951, p=0.032*
  Our: 0.90±0.04% vs Paper: 0.95%
(50, 5, False, 'original'): t=-1.340, p=0.238
  Our: 0.93±0.04% vs Paper: 0.95%
(50, 15, False, 'original'): t=-17.197, p=0.000***
  Our: 0.96±0.00% vs Paper: 0.97%

--- IoU ---
(10, 2, False, 'original'): t=5.101, p=0.004**
  Our: 0.21±0.03% vs 

# Holm-Bonferroni

In [98]:
from statsmodels.stats.multitest import multipletests
import pandas as pd

# Collect all p-values and their corresponding information
all_tests = []
for metric in metrics:
    for idx, result in t_test_results[metric].items():
        all_tests.append({
            'metric': metric,
            'configuration': idx,
            'p_value': result['p_value'],
            't_statistic': result['t_statistic'],
            'our_mean': result['our_mean'],
            'paper_mean': result['paper_mean'],
            'our_std': result['our_std']
        })

# Extract p-values for correction
p_values = [test['p_value'] for test in all_tests]

# Apply Holm-Bonferroni correction
rejected, p_corrected, alpha_sidak, alpha_bonf = multipletests(
    p_values, 
    alpha=0.05, 
    method='holm'
)

# Add corrected results back to our test data
for i, test in enumerate(all_tests):
    test['p_corrected'] = p_corrected[i]
    test['rejected_holm'] = rejected[i]

print("HOLM-BONFERRONI CORRECTED RESULTS")
print("="*60)
print("H0: Our results = Paper results")
print("H1: Our results ≠ Paper results")
print(f"Total number of tests: {len(p_values)}")
print(f"Family-wise error rate (FWER) controlled at α = 0.05")
print("\nSignificance levels: * p_corrected<0.05, ** p_corrected<0.01, *** p_corrected<0.001")
print("\nResults (t-statistic, p_original → p_corrected, significant):")

# Display results organized by metric
for metric in metrics:
    print(f"\n--- {metric} ---")
    
    metric_tests = [test for test in all_tests if test['metric'] == metric]
    
    for test in metric_tests:
        # Determine significance level for corrected p-value
        if test['p_corrected'] < 0.001:
            sig = "***"
        elif test['p_corrected'] < 0.01:
            sig = "**"
        elif test['p_corrected'] < 0.05:
            sig = "*"
        else:
            sig = ""
        
        # Show if hypothesis was rejected
        rejected_status = "SIGNIFICANT" if test['rejected_holm'] else "not significant"
        
        print(f"{test['configuration']}:")
        print(f"  t={test['t_statistic']:.3f}, p={test['p_value']:.3f} → p_corrected={test['p_corrected']:.3f}{sig}")
        print(f"  Our: {test['our_mean']*100:.2f}±{test['our_std']*100:.2f}% vs Paper: {test['paper_mean']*100:.2f}%")
        print(f"  Result: {rejected_status}")

# Summary statistics
print("\n" + "="*60)
print("SUMMARY - HOLM-BONFERRONI CORRECTED")
print("="*60)

total_significant_original = sum(1 for test in all_tests if test['p_value'] < 0.05)
total_significant_corrected = sum(1 for test in all_tests if test['rejected_holm'])

print(f"Total tests performed: {len(all_tests)}")
print(f"Significant before correction (p<0.05): {total_significant_original}")
print(f"Significant after Holm-Bonferroni correction: {total_significant_corrected}")
print(f"Reduction due to multiple comparisons correction: {total_significant_original - total_significant_corrected}")

# Summary by metric
for metric in metrics:
    metric_tests = [test for test in all_tests if test['metric'] == metric]
    original_sig = sum(1 for test in metric_tests if test['p_value'] < 0.05)
    corrected_sig = sum(1 for test in metric_tests if test['rejected_holm'])
    
    print(f"\n{metric}:")
    print(f"  Total comparisons: {len(metric_tests)}")
    print(f"  Significant before correction: {original_sig}")
    print(f"  Significant after correction: {corrected_sig}")
    print(f"  Proportion significant after correction: {corrected_sig/len(metric_tests):.1%}")

# Create a summary DataFrame for easy viewing
summary_df = pd.DataFrame(all_tests)
summary_df['configuration_str'] = summary_df['configuration'].astype(str)
summary_df['significant_original'] = summary_df['p_value'] < 0.05
summary_df['significant_corrected'] = summary_df['rejected_holm']

print(f"\nDetailed results saved in 'summary_df' DataFrame with columns:")
print(f"  {list(summary_df.columns)}")

# Optional: Save detailed results to CSV
summary_df.to_csv('holm_bonferroni_results.csv', index=False)
print(f"\nDetailed results saved to 'holm_bonferroni_results.csv'")

HOLM-BONFERRONI CORRECTED RESULTS
H0: Our results = Paper results
H1: Our results ≠ Paper results
Total number of tests: 27
Family-wise error rate (FWER) controlled at α = 0.05

Significance levels: * p_corrected<0.05, ** p_corrected<0.01, *** p_corrected<0.001

Results (t-statistic, p_original → p_corrected, significant):

--- Accuracy ---
(10, 2, False, 'original'):
  t=0.146, p=0.889 → p_corrected=1.000
  Our: 93.37±0.71% vs Paper: 93.33%
  Result: not significant
(10, 5, False, 'original'):
  t=-0.931, p=0.394 → p_corrected=1.000
  Our: 94.93±0.29% vs Paper: 95.04%
  Result: not significant
(10, 15, False, 'original'):
  t=9.427, p=0.000 → p_corrected=0.006**
  Our: 96.06±0.11% vs Paper: 95.64%
  Result: SIGNIFICANT
(20, 2, False, 'original'):
  t=-2.235, p=0.076 → p_corrected=0.908
  Our: 93.16±1.18% vs Paper: 94.23%
  Result: not significant
(20, 5, False, 'original'):
  t=-5.168, p=0.004 → p_corrected=0.068
  Our: 95.25±0.18% vs Paper: 95.63%
  Result: not significant
(20, 15, F

# Results Tables

In [100]:
# Reshape results back to original table format
def create_results_table(all_tests, value_column, metrics):
    """Helper function to create a table with original indexing"""
    results_dict = {}
    
    for test in all_tests:
        config = test['configuration']
        metric = test['metric']
        value = test[value_column]
        
        if config not in results_dict:
            results_dict[config] = {}
        results_dict[config][metric] = value
    
    # Convert to DataFrame with proper indexing
    results_df = pd.DataFrame.from_dict(results_dict, orient='index')
    results_df = results_df.reindex(columns=metrics)  # Ensure column order
    return results_df

# Create tables for different aspects of the results
metrics = ['Accuracy', 'IoU', 'AP']

# Table 1: Original p-values
p_values_table = create_results_table(all_tests, 'p_value', metrics)

# Table 2: Holm-Bonferroni corrected p-values
p_corrected_table = create_results_table(all_tests, 'p_corrected', metrics)

# Table 3: T-statistics
t_stats_table = create_results_table(all_tests, 't_statistic', metrics)

# Table 4: Significance status (True = significant after correction)
significance_table = create_results_table(all_tests, 'rejected_holm', metrics)

print("ORIGINAL P-VALUES")
print("="*50)
print(p_values_table.round(4))

print("\n\nHOLM-BONFERRONI CORRECTED P-VALUES")
print("="*50)
print(p_corrected_table.round(4))

print("\n\nT-STATISTICS")
print("="*50)
print(t_stats_table.round(3))

print("\n\nSIGNIFICANCE STATUS (after Holm-Bonferroni correction)")
print("="*50)
print("True = Significant difference from paper results")
print(significance_table)

# Create a combined summary table showing key information
print("\n\nCOMBINED SUMMARY TABLE")
print("="*50)
combined_summary = pd.DataFrame(index=p_values_table.index)

for metric in metrics:
    # Format: "p_orig → p_corr (sig_status)"
    combined_summary[f'{metric}_p_original'] = p_values_table[metric].round(4)
    combined_summary[f'{metric}_p_corrected'] = p_corrected_table[metric].round(4)
    combined_summary[f'{metric}_significant'] = significance_table[metric]
    
    # Create a readable summary column
    combined_summary[f'{metric}_summary'] = combined_summary.apply(
        lambda row: f"{row[f'{metric}_p_original']:.3f} → {row[f'{metric}_p_corrected']:.3f} {'*' if row[f'{metric}_significant'] else ''}", 
        axis=1
    )

# Show just the summary columns for readability
summary_columns = [f'{metric}_summary' for metric in metrics]
print(combined_summary[summary_columns])

# Create a table showing differences from paper (Our - Paper)
print("\n\nDIFFERENCES FROM PAPER RESULTS (Our - Paper)")
print("="*50)
differences_table = pd.DataFrame(index=paper_df.index)

for metric in metrics:
    # Calculate differences (convert means back to same scale as paper)
    our_values = means.loc[paper_df.index, metric] * 100  # Convert to percentage
    paper_values = paper_df[metric]
    differences_table[metric] = our_values - paper_values

print(differences_table.round(2))

# Mark significant differences with asterisks
print("\n\nDIFFERENCES WITH SIGNIFICANCE MARKERS")
print("="*50)
print("* = Significant difference after Holm-Bonferroni correction")

differences_with_sig = differences_table.copy()
for metric in metrics:
    differences_with_sig[metric] = differences_with_sig[metric].round(2).astype(str)
    
    # Add asterisks for significant differences
    sig_mask = significance_table[metric]
    differences_with_sig.loc[sig_mask, metric] = differences_with_sig.loc[sig_mask, metric] + '*'

print(differences_with_sig)

# Save all tables
p_values_table.to_csv('p_values_original.csv')
p_corrected_table.to_csv('p_values_holm_corrected.csv')
t_stats_table.to_csv('t_statistics.csv')
significance_table.to_csv('significance_status.csv')
differences_with_sig.to_csv('differences_from_paper.csv')

print(f"\n\nAll tables saved as CSV files:")
print("- p_values_original.csv")
print("- p_values_holm_corrected.csv") 
print("- t_statistics.csv")
print("- significance_status.csv")
print("- differences_from_paper.csv")

ORIGINAL P-VALUES
                      Accuracy     IoU      AP
10 2  False original    0.8894  0.0038  0.0010
   5  False original    0.3944  0.1221  0.2873
   15 False original    0.0002  0.7711  0.0047
20 2  False original    0.0757  0.5423  0.9718
   5  False original    0.0036  0.0036  0.0029
   15 False original    0.0014  0.0087  0.0031
50 2  False original    0.0318  0.0177  0.1019
   5  False original    0.2378  0.3596  0.2846
   15 False original    0.0000  0.0001  0.0006


HOLM-BONFERRONI CORRECTED P-VALUES
                      Accuracy     IoU      AP
10 2  False original    1.0000  0.0677  0.0236
   5  False original    1.0000  1.0000  1.0000
   15 False original    0.0057  1.0000  0.0750
20 2  False original    0.9079  1.0000  1.0000
   5  False original    0.0677  0.0677  0.0601
   15 False original    0.0297  0.1300  0.0612
50 2  False original    0.4138  0.2477  1.0000
   5  False original    1.0000  1.0000  1.0000
   15 False original    0.0003  0.0020  0.0153


T-S